# Notebook 02 - Anatomia del Prompt

## Objetivos
- Identificar los componentes de un prompt efectivo.
- Comparar prompts malos vs buenos con ejemplos reales.
- Construir prompts estructurados con rol, contexto, instruccion y formato.

## Introduccion
Un prompt es la interfaz entre tu y el LLM. Un prompt vago produce respuestas vagas. Este notebook ensena a diseñar prompts profesionales con anatomia completa.

In [ ]:
from pathlib import Path
from IPython.display import display, Markdown
import pandas as pd
import matplotlib.pyplot as plt

BASE = Path('..')
DATASETS = BASE / 'datasets'
print('Entorno listo. Datasets:', list(DATASETS.glob('*.csv')))

## 1) Componentes de un buen prompt

In [ ]:
anatomia = pd.DataFrame([
    {'componente': 'Rol', 'funcion': 'Define quien es el modelo', 'ejemplo': 'Eres un analista de datos senior...'},
    {'componente': 'Contexto', 'funcion': 'Informacion de fondo', 'ejemplo': 'La empresa vende software B2B...'},
    {'componente': 'Instruccion', 'funcion': 'Que tarea realizar', 'ejemplo': 'Clasifica el siguiente ticket...'},
    {'componente': 'Restricciones', 'funcion': 'Que NO hacer', 'ejemplo': 'No inventes datos. Max 50 palabras'},
    {'componente': 'Formato', 'funcion': 'Como estructurar la salida', 'ejemplo': 'Responde en JSON'},
    {'componente': 'Ejemplos', 'funcion': 'Demostrar el patron (few-shot)', 'ejemplo': 'Ticket: ... -> alta'},
    {'componente': 'Entrada', 'funcion': 'Dato a procesar', 'ejemplo': 'Ticket: No puedo acceder'},
])
display(anatomia)

## 2) Prompts malos vs buenos (CSV del curso)

In [ ]:
df_prompts = pd.read_csv(DATASETS / 'prompts_buenos_malos.csv')
display(df_prompts)

## 3) Funcion para construir prompts estructurados

In [ ]:
def construir_prompt(rol='', contexto='', instruccion='', restricciones='', formato='', ejemplos='', entrada=''):
    partes = []
    if rol:
        partes.append(rol)
    if contexto:
        partes.append(f'Contexto:\n{contexto}')
    if instruccion:
        partes.append(f'Tarea: {instruccion}')
    if restricciones:
        partes.append(f'Reglas: {restricciones}')
    if formato:
        partes.append(f'Formato de salida: {formato}')
    if ejemplos:
        partes.append(f'Ejemplos:\n{ejemplos}')
    if entrada:
        partes.append(f'Entrada: {entrada}')
    partes.append('Respuesta:')
    return '\n\n'.join(partes)

print('Funcion construir_prompt() lista')

## 4) Ejemplo: prompt estructurado para clasificacion

In [ ]:
prompt_clasificacion = construir_prompt(
    rol='Eres un analista de soporte al cliente.',
    instruccion='Clasifica el sentimiento como positivo, negativo o neutral.',
    restricciones='Responde SOLO con la etiqueta. Sin explicacion.',
    entrada='El producto llegó roto y nadie respondió a mis correos.',
)
print(prompt_clasificacion)

## 5) Probar el prompt con GPT-2 en español

In [ ]:
from transformers import pipeline, set_seed

set_seed(42)
generator = pipeline('text-generation', model='datificate/gpt2-small-spanish')
print('GPT-2 en español listo para experimentos de prompting')

In [ ]:
def generar(prompt, max_new_tokens=15, temperature=0.3):
    out = generator(
        prompt,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        pad_token_id=generator.tokenizer.eos_token_id,
    )
    return out[0]['generated_text']

print('Funcion generar() lista')

In [ ]:
resultado = generar(prompt_clasificacion, max_new_tokens=10, temperature=0.3)
print(resultado)

## 6) Ejercicio en vivo: mejora un prompt malo

In [ ]:
prompt_malo = 'Analiza este texto sobre atencion al cliente'
prompt_mejorado = construir_prompt(
    rol='Eres un analista de calidad de soporte.',
    instruccion='Califica la atencion como excelente, buena o mala.',
    restricciones='Una sola palabra. Sin explicacion.',
    formato='Etiqueta unica',
    entrada='Espere 45 minutos en linea y el agente fue grosero.',
)

comparacion = pd.DataFrame([
    {'version': 'Malo', 'prompt': prompt_malo},
    {'version': 'Mejorado', 'prompt': prompt_mejorado},
])
for _, row in comparacion.iterrows():
    print(f"\n=== {row['version']} ===")
    print(row['prompt'][:200])
    print('--- Generado ---')
    print(generar(row['prompt'], max_new_tokens=15, temperature=0.5))

## Resultados
Identificamos 7 componentes del prompt, comparamos versiones malas/buenas y construimos prompts estructurados con una funcion reutilizable.

## Conclusiones
La calidad del prompt determina la calidad de la salida. Invertir 5 minutos en diseñar el prompt ahorra horas de post-procesamiento.

## Ejercicios guiados resueltos
**Ejercicio:** Construye un prompt para resumir documentos con los 7 componentes.

**Solucion:**

In [ ]:
prompt_resumen = construir_prompt(
    rol='Eres asistente ejecutivo.',
    instruccion='Resume el documento en exactamente 3 viñetas.',
    restricciones='Solo hechos del texto. Maximo 15 palabras por viñeta.',
    formato='Lista con guiones',
    entrada='Las ventas del Q3 alcanzaron 2.4M USD con 18% de crecimiento. B2B fue 62% del total.',
)
print(prompt_resumen)

## Ejercicios propuestos
1. Mejora 3 prompts malos del CSV.
2. Diseña un prompt con delimitadores XML (<context>, <task>).
3. Crea una plantilla reutilizable para tu industria.

## Preguntas de reflexion
1. Que componente tiene mayor impacto en la precision?
2. Por que las restricciones son tan importantes?
3. Como documentarias prompts en un equipo de 10 personas?